# 10 - Mini Project: Train And Explain A Tiny Classifier

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

from course.checks import qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 1. 项目地图

| 步骤 | 你要产出什么 |
|---|---|
| 数据 | `xs`, `ys` |
| 模型 | `MLP(2, [4, 1])` |
| 预测 | 每个样本一个 score |
| loss | 平方误差或 hinge loss |
| 训练 | 多轮 zero_grad/backward/update |
| 评估 | accuracy 和最终 loss |
| 解释 | 能说清每一步的作用 |

## 2. 数据和模型

In [ ]:
project_xs = [
    [-2.0, -1.0],
    [-1.0, -1.0],
    [-1.0,  2.0],
    [ 1.0,  1.0],
    [ 2.0, -1.0],
    [ 2.0,  2.0],
]
project_ys = [-1, -1, 1, 1, 1, 1]

xs = project_xs
ys = project_ys

random.seed(42)
model = MLP(2, [4, 1])
print(model)
print('parameters:', len(model.parameters()))

for x, y in zip(xs, ys):
    print(x, '->', y)

## 3. 写 predict、loss、accuracy

这个小项目用平方误差：

$$
loss = \frac{1}{N}\sum_i (score_i - y_i)^2
$$

分类时再看：

```text
score > 0 -> 预测 1
score < 0 -> 预测 -1
```

In [ ]:
def predict(xrow):
    return model([Value(x) for x in xrow])


def loss():
    scores = [predict(x) for x in xs]
    losses = [(score - y) ** 2 for score, y in zip(scores, ys)]
    total = sum(losses) * (1.0 / len(losses))
    return total, scores


def label_from_score(score):
    return 1 if score > 0 else -1


def accuracy(scores, labels):
    preds = [label_from_score(score.data) for score in scores]
    correct = sum(int(pred == y) for pred, y in zip(preds, labels))
    return correct / len(labels), preds

initial_loss, initial_scores = loss()
initial_acc, initial_preds = accuracy(initial_scores, ys)
print('initial loss:', initial_loss.data)
print('initial acc :', initial_acc)
print('initial preds:', initial_preds)

## 4. 训练小项目

这里训练 100 步。你要观察三件事：

```text
loss 是否下降
accuracy 是否提升
predictions 是否更接近标签
```

In [ ]:
learning_rate = 0.05
history = []

for step in range(100):
    total_loss, scores = loss()
    acc, preds = accuracy(scores, ys)
    history.append(total_loss.data)

    model.zero_grad()
    total_loss.backward()

    for p in model.parameters():
        p.data += -learning_rate * p.grad

    if step % 20 == 0 or step == 99:
        print(f'step {step:03d} loss={total_loss.data:.5f} acc={acc:.2f} preds={preds}')

final_loss, final_scores = loss()
final_acc, final_preds = accuracy(final_scores, ys)
print()
print('initial loss:', history[0])
print('final loss  :', final_loss.data)
print('final acc   :', final_acc)
print('final preds :', final_preds)
assert final_loss.data < history[0]

## 5. 看结果，不只看数字

如果能画图，就画出训练后的分界区域。这个图是给直觉用的：蓝色区域预测 `1`，红色区域预测 `-1`。

In [ ]:
if MATPLOTLIB_AVAILABLE:
    grid_x = [i / 10 for i in range(-30, 31)]
    grid_y = [i / 10 for i in range(-30, 31)]
    Z = []
    for yy in grid_y:
        row = []
        for xx in grid_x:
            row.append(predict([xx, yy]).data)
        Z.append(row)

    plt.figure(figsize=(5, 4))
    plt.contourf(grid_x, grid_y, Z, levels=[-999, 0, 999], alpha=0.25, colors=['#ff9999', '#99ccff'])
    plt.contour(grid_x, grid_y, Z, levels=[0], colors='black', linewidths=2)
    for x, y in zip(xs, ys):
        color = '#1f77b4' if y == 1 else '#d62728'
        marker = 'o' if y == 1 else 'x'
        plt.scatter(x[0], x[1], c=color, marker=marker, s=80)
    plt.title('Mini project classifier')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.grid(True, alpha=0.2)
    plt.show()
else:
    print('matplotlib 不可用，跳过画图。')

## What To Remember

```text
1. 一个项目不是只有 model，还包括数据、loss、训练、评估和解释。
2. score 是模型输出，label 是目标答案。
3. loss 用来训练，accuracy 用来评价分类结果。
4. loss.backward() 只算梯度，真正改变参数的是 update。
5. 能解释整条链路，比只看到 loss 下降更重要。
```

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - 项目主链路排序

In [ ]:
# 填写说明：
# - 填一个列表：项目训练主流程顺序，包含 forward/loss/backward/update。
# - 填完后运行本 cell，看 qa_check 的提示。

student_order = None  # ['forward','loss','backward','update']



qa_check('qa_check_class_10_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 沿项目主链路排查：数据画出来，score 转 label，loss 反传 grad，再看参数更新和边界图是否一致。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_order` 填 `['forward'`。

</details>

## Run - 先不用 MLP，用一条手写边界看分类

In [ ]:
def simple_score(point):
    x1, x2 = point
    return x1 - x2

points = [(-2,-1), (2,1)]
for p in points:
    score = simple_score(p)
    label = 1 if score > 0 else -1
    print(p, 'score=', score, 'label=', label)

## Run - 一个小 MLP 的参数数量

In [ ]:
from micrograd.nn import MLP
random.seed(7)
model = MLP(2, [4, 1])
print(model)
print('parameters:', len(model.parameters()))

## Modify - 改隐藏层大小

In [ ]:
# 填写说明：
# - 填一个数字：MLP(2,[3,1]) 的总参数数量。
# - 填完后运行本 cell，看 qa_check 的提示。

# MLP(2,[3,1]) 参数数量是多少？
student_param_count = None



qa_check('qa_check_class_10_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 沿项目主链路排查：数据画出来，score 转 label，loss 反传 grad，再看参数更新和边界图是否一致。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_param_count` 填 `13`。

</details>